# Практика: FinOps — архитектура, управляемая стоимостью (Emulator)

**Цель:** Пройти путь от получения cloud bill до архитектурного решения, снизив TCO AI-системы **≥ 20%** без деградации качества.

**Вход:**
- упрощённый cloud bill
- описание архитектуры AI-системы (training pipeline + realtime inference + S3 + K8s + логи)

**Что демонстрируем:**
1. Cost = f(Architecture, Load, Efficiency) — считаем это в коде.
2. Архитектор = cost owner. Каждое решение = архитектурный рычаг с cost-эффектом.
3. Минимум 20% экономии через 3–5 архитектурных изменений.

### Шаг 1. Импорты и базовая модель
Никаких облачных SDK — весь cloud bill и архитектура эмулируются локально, чтобы студенты могли менять параметры и сразу видеть цифры.

In [22]:
from dataclasses import dataclass, field, replace
from typing import List, Dict, Callable
from copy import deepcopy

@dataclass
class CostItem:
    service: str          # строка cloud bill
    category: str         # compute / storage / network / observability
    monthly_cost: float   # $/month
    note: str = ""        # архитектурная причина стоимости

    def __repr__(self):
        return f"{self.service:<20} {self.category:<14} ${self.monthly_cost:>8,.0f}  — {self.note}"

@dataclass
class CloudBill:
    items: List[CostItem]

    @property
    def total(self) -> float:
        return sum(i.monthly_cost for i in self.items)

    def print(self, title: str = "Cloud bill"):
        print(f"\n=== {title} ===")
        for it in self.items:
            print(it)
        print("-" * 70)
        print(f"{'TOTAL':<20} {'':<14} ${self.total:>8,.0f}/month")

### Шаг 2. Исходные данные: cloud bill + архитектура
Тот самый bill, который мы видели на слайдах 20–21. Архитектура **намеренно плохая** — задача её исправить.

In [23]:
BASELINE = CloudBill([
    CostItem("EC2 (GPU)",        "compute",       12000, "4x p3.8xlarge, работают 24/7"),
    CostItem("S3",                "storage",        3500, "всё в Standard, нет lifecycle policy"),
    CostItem("Data transfer",     "network",        2000, "cross-AZ + egress из региона"),
    CostItem("Kubernetes nodes",  "compute",        4000, "фикс 8 нод m5.2xlarge, без HPA"),
    CostItem("Logging",           "observability",  1500, "DEBUG-логи всех сервисов в CloudWatch"),
])

ARCHITECTURE = {
    "training":   {"schedule": "daily",    "gpu_hours": 720, "note": "ежедневный переобучающий запуск"},
    "inference":  {"schedule": "realtime", "gpu_hours": 720, "note": "realtime даже ночью, когда трафика нет"},
    "storage":    {"total_gb": 40000,      "hot_ratio": 1.0, "note": "100% в hot storage"},
    "k8s":        {"nodes": 8,             "utilization": 0.25, "note": "средняя утилизация 25%"},
    "logging":    {"level": "DEBUG",       "gb_per_day": 50,    "note": "логируем всё подряд"},
}

BASELINE.print("Baseline cloud bill")
print("\nАрхитектура:")
for k, v in ARCHITECTURE.items():
    print(f"  {k:<10} {v}")


=== Baseline cloud bill ===
EC2 (GPU)            compute        $  12,000  — 4x p3.8xlarge, работают 24/7
S3                   storage        $   3,500  — всё в Standard, нет lifecycle policy
Data transfer        network        $   2,000  — cross-AZ + egress из региона
Kubernetes nodes     compute        $   4,000  — фикс 8 нод m5.2xlarge, без HPA
Logging              observability  $   1,500  — DEBUG-логи всех сервисов в CloudWatch
----------------------------------------------------------------------
TOTAL                               $  23,000/month

Архитектура:
  training   {'schedule': 'daily', 'gpu_hours': 720, 'note': 'ежедневный переобучающий запуск'}
  inference  {'schedule': 'realtime', 'gpu_hours': 720, 'note': 'realtime даже ночью, когда трафика нет'}
  storage    {'total_gb': 40000, 'hot_ratio': 1.0, 'note': '100% в hot storage'}
  k8s        {'nodes': 8, 'utilization': 0.25, 'note': 'средняя утилизация 25%'}
  logging    {'level': 'DEBUG', 'gb_per_day': 50, 'note': 'ло

### Шаг 3. Задание
Перед тем как смотреть код дальше — ответьте на три вопроса:

1. **Сколько стоит?** — какие 3 строки bill дают >70% затрат?
2. **Можно ли дешевле?** — какое архитектурное решение в `ARCHITECTURE` отвечает за каждую из этих строк?
3. **Что будет при росте нагрузки?** — какое из решений плохо масштабируется?

Подсказки:
- GPU используется постоянно?
- нужен ли realtime?
- можно ли batching?
- нужен ли hot storage?
- есть ли autoscaling?

### Шаг 4. Архитектурные рычаги (FinOps levers)
Каждый рычаг — чистая функция `CloudBill -> CloudBill`. 

Это важно: в реальности каждое изменение = отдельный PR + runbook. В эмуляторе - отдельный метод с явным cost-эффектом.

Для оценки экономии взяты консервативные оценки, не маркетинговые.

In [24]:
Lever = Callable[[CloudBill], CloudBill]

def apply_discount(bill: CloudBill, service: str, factor: float, new_note: str) -> CloudBill:
    """Уменьшает monthly_cost строки bill на (1 - factor). factor = доля после оптимизации."""
    new_items = []
    for it in bill.items:
        if it.service == service:
            new_items.append(replace(it, monthly_cost=it.monthly_cost * factor, note=new_note))
        else:
            new_items.append(it)
    return CloudBill(new_items)

def gpu_to_spot(bill: CloudBill) -> CloudBill:
    """GPU -> Spot + scheduling. Экономия до 60% (слайд 25)."""
    return apply_discount(bill, "EC2 (GPU)", 0.40, "Spot + scheduling по окну трафика")

def training_batch(bill: CloudBill) -> CloudBill:
    """Training ежедневно - batch реже. Часть compute-часов убираем. Экономия ~25% сверху GPU-бюджета."""
    # training — это часть той же GPU-строки; режем ещё 25% оставшегося.
    return apply_discount(bill, "EC2 (GPU)",
                          factor=1.0 * 0.75,  # -25% от текущей стоимости строки
                          new_note="batch training 2x/неделю вместо 7x") if False else \
           CloudBill([
               replace(it, monthly_cost=it.monthly_cost * 0.75,
                       note=it.note + " + batch training 2x/нед")
               if it.service == "EC2 (GPU)" else it
               for it in bill.items
           ])

def s3_lifecycle(bill: CloudBill) -> CloudBill:
    """S3 lifecycle hot->cold для старых данных. Экономия 50–80% storage (слайд 25)."""
    return apply_discount(bill, "S3", 0.35, "lifecycle: >30 дней -> Glacier IA")

def k8s_autoscaling(bill: CloudBill) -> CloudBill:
    """Cluster autoscaler + HPA вместо фиксированных 8 нод. Экономия ~30%."""
    return apply_discount(bill, "Kubernetes nodes", 0.70, "HPA + cluster autoscaler, min=2")

def log_reduction(bill: CloudBill) -> CloudBill:
    """DEBUG -> INFO, sampling, dropping healthchecks. Экономия ~50%."""
    return apply_discount(bill, "Logging", 0.50, "INFO-уровень + sampling 1/10 для access-логов")

def network_locality(bill: CloudBill) -> CloudBill:
    """Убираем cross-AZ трафик, кэшируем egress через CloudFront. ~40%."""
    return apply_discount(bill, "Data transfer", 0.60, "single-AZ для training, CDN для egress")

LEVERS: Dict[str, Lever] = {
    "GPU -> Spot":             gpu_to_spot,
    "Training -> batch":       training_batch,
    "S3 lifecycle":           s3_lifecycle,
    "K8s autoscaling":        k8s_autoscaling,
    "Logging reduction":      log_reduction,
    "Network locality":       network_locality,
}

### Шаг 5. FinOps-отчёт: baseline -> оптимизация -> экономия
Применяем набор шагов к уменьшению и печатаем отчёт, как сделали бы в реальном review: что изменили, насколько это дёшево, хватает ли до цели в 20%.

In [25]:
def apply_levers(bill: CloudBill, lever_names: List[str]) -> CloudBill:
    current = deepcopy(bill)
    for name in lever_names:
        current = LEVERS[name](current)
    return current

def finops_report(baseline: CloudBill, optimized: CloudBill, applied: List[str], target_saving: float = 0.20):
    print("\n" + "=" * 70)
    print("FINOPS REPORT")
    print("=" * 70)
    print(f"Применённые рычаги: {', '.join(applied)}")
    print()
    print(f"{'Service':<20} {'Before':>10} {'After':>10} {'Δ':>10}")
    print("-" * 70)
    for b, o in zip(baseline.items, optimized.items):
        delta = o.monthly_cost - b.monthly_cost
        print(f"{b.service:<20} ${b.monthly_cost:>8,.0f}  ${o.monthly_cost:>8,.0f}  ${delta:>+8,.0f}")
    print("-" * 70)
    saving_abs = baseline.total - optimized.total
    saving_pct = saving_abs / baseline.total
    print(f"{'TOTAL':<20} ${baseline.total:>8,.0f}  ${optimized.total:>8,.0f}  ${-saving_abs:>+8,.0f}")
    print(f"\nЭкономия: {saving_pct*100:5.1f}%  (${saving_abs:,.0f}/month, ${saving_abs*12:,.0f}/year)")
    ok = saving_pct >= target_saving
    print(f"Цель ≥ {target_saving*100:.0f}%: {'✅ достигнута' if ok else '❌ не достигнута — добавьте ещё рычаг'}")

# Демонстрационный набор: 5 рычагов воздействия
applied = ["GPU -> Spot", "S3 lifecycle", "K8s autoscaling", "Logging reduction", "Network locality"]
optimized = apply_levers(BASELINE, applied)

BASELINE.print("Baseline")
optimized.print("After optimizations")
finops_report(BASELINE, optimized, applied)


=== Baseline ===
EC2 (GPU)            compute        $  12,000  — 4x p3.8xlarge, работают 24/7
S3                   storage        $   3,500  — всё в Standard, нет lifecycle policy
Data transfer        network        $   2,000  — cross-AZ + egress из региона
Kubernetes nodes     compute        $   4,000  — фикс 8 нод m5.2xlarge, без HPA
Logging              observability  $   1,500  — DEBUG-логи всех сервисов в CloudWatch
----------------------------------------------------------------------
TOTAL                               $  23,000/month

=== After optimizations ===
EC2 (GPU)            compute        $   4,800  — Spot + scheduling по окну трафика
S3                   storage        $   1,225  — lifecycle: >30 дней -> Glacier IA
Data transfer        network        $   1,200  — single-AZ для training, CDN для egress
Kubernetes nodes     compute        $   2,800  — HPA + cluster autoscaler, min=2
Logging              observability  $     750  — INFO-уровень + sampling 1/10 для acce

### Шаг 6. Sensitivity: что если нагрузка вырастет ×3?
Слайд 27 требует третий ответ: *что будет при росте нагрузки*. Проверим оба варианта на load-multiplier.

In [26]:
def scale_bill(bill: CloudBill, load_multiplier: float, elastic_categories=("compute", "network", "observability")) -> CloudBill:
    """Эластичные категории масштабируются с нагрузкой, storage — почти линейно по времени, не по нагрузке."""
    scaled = []
    for it in bill.items:
        mult = load_multiplier if it.category in elastic_categories else 1.0 + (load_multiplier - 1) * 0.2
        scaled.append(replace(it, monthly_cost=it.monthly_cost * mult))
    return CloudBill(scaled)

for load in [1.0, 2.0, 3.0]:
    b = scale_bill(BASELINE, load)
    o = scale_bill(optimized, load)
    diff_pct = (b.total - o.total) / b.total * 100
    print(f"Load ×{load:>3.1f}:  baseline ${b.total:>8,.0f}  optimized ${o.total:>8,.0f}  saving {diff_pct:4.1f}%")

Load ×1.0:  baseline $  23,000  optimized $  10,775  saving 53.2%
Load ×2.0:  baseline $  43,200  optimized $  20,570  saving 52.4%
Load ×3.0:  baseline $  63,400  optimized $  30,365  saving 52.1%


### Что мы продемонстрировали этим кодом

1. **Cost = f(Architecture, Load, Efficiency).** Bill — следствие, шаги снижения стоимости лежат в архитектуре (`ARCHITECTURE` dict), а не в финансовом отделе.
2. **Visibility.** Каждая строка bill имеет `note` — архитектурную причину стоимости. Без этого FinOps невозможен (правило: *нет тегов -> нет FinOps*).
3. **Accountability.** Каждый рычаг = отдельная функция с предсказуемым эффектом. Это и есть единица ответственности архитектора — не «снизить cloud bill», а «перевести GPU на Spot».
4. **Optimization.** Пять рычагов из слайда 24 дают суммарную экономию ≥20% без потери функциональности.
5. **Continuous improvement.** `scale_bill` показывает, что optimized-архитектура **продолжает экономить** при росте нагрузки — оптимизация не одноразовая.

### Задание

1. Уберите из `applied` самый «жирный» рычаг (`GPU -> Spot`). Достигнется ли цель 20%? Какие рычаги надо добавить взамен?
2. Добавьте свой рычаг `model_distillation` в `LEVERS`: замена большой модели на distilled-версию даёт -40% GPU при -2% качества. Где его место в иерархии?
3. Измените `ARCHITECTURE['inference']['schedule']` на `batch`. Какой рычаг тогда станет ненужным, а какой — обязательным?
4. Постройте график saving(%) в зависимости от load_multiplier ∈ [0.5, 5.0]. Где точка, после которой optimized-вариант перестаёт быть выгодным? (спойлер: её нет — в этом и смысл архитектурной оптимизации).

#### Решение

##### 1. Убираем GPU -> Spot — достигнется ли 20%?             
Считаем экономию без самого «жирного» рычага:

| Рычаг | Экономия |
| -- | -- |
| S3 lifecycle | $2,275 |
| K8s autoscaling | $1,200 |
| Network locality | $800 |
| Logging reduction | $750 |
| Итого | $5,025 / $23,000 = 21.8% |

Цель формально достигается (21.8%), но впритык — любое подорожание услуг и цель срывается. В реальности это нездоровый результат: CFO услышит «20%» и успокоится, а главный источник расходов (GPU $12k, 52% bill) останется нетронутым.

Что добавить взамен: 
  - Training -> batch (−25% от GPU-строки = $3,000/мес) — даст 34.9% в сумме.
  - Ещё лучше — новый рычаг model_distillation (см. п. 2): −40% от GPU = $4,800/мес. 

Принцип: нельзя экономить на мелочёвке, игнорируя самую большую строку bill. Это нарушение правила «80/20» в FinOps.

##### 2. model_distillation (−40% GPU при −2% качества) — где его место?

Место в иерархии — САМЫЙ ВЕРХ, выше Spot и batch. Иерархия архитектурных рычагов по FinOps:

  1. Reduce need — изменить сам workload (distillation, quantization, выбор модели поменьше). Это архитектурное решение.
  2. Change pattern — batch вместо realtime, scheduling, HPA (меняем как используем ресурс).
  3. Change pricing — Spot, Reserved, Savings Plans (меняем как платим).
  4. Clean up waste — lifecycle, log levels, cross-AZ (убираем явный мусор).
                                                                   
model_distillation — уровень №1: мы уменьшаем потребность в GPU, а не оптимизируем её утилизацию. После distillation и Spot, и batch дают меньше абсолютной экономии (потому что база GPU уже меньше), но процент сохраняется — значит решение устойчивое.

Caveat: −2% качества — это архитектурный trade-off, требующий согласия product-owner, не единоличное решение FinOps-инженера.

##### 3. ARCHITECTURE['inference']['schedule'] = 'batch'
Ненужный рычаг: K8s autoscaling (HPA).
HPA нужен, когда нагрузка непредсказуема и пиковая (realtime-трафик). При batch-inference расписание известно заранее — можно поднять фиксированный pool под окно расчёта и потушить после. Autoscaler становится overhead'ом.

Обязательный рычаг: GPU -> Spot.
Batch-workload идеально совместим с Spot: можно перезапустить задачу при вытеснении, SLA по latency нет. Для realtime-inference Spot рискован (interruption = downtime для пользователя), поэтому в исходной архитектуре это был компромисс. При batch — это буквально бесплатная экономия 60% на GPU.

Бонусом: Training -> batch перестаёт быть отдельным рычагом — training и inference объединяются в один batch-pipeline, можно переиспользовать GPU-pool между ними.

##### 4. Saving(%) как функция от load_multiplier
Аналитически (из scale_bill): эластичные категории (compute+network+observability) масштабируются линейно, storage — почти константен (коэф. 0.2).

- $Baseline(L) = 19 500\cdot L + 3 500 \cdot (1 + 0.2\cdot (L−1)) = 20 200 \cdot L + 2 800$
- $Optimized(L) = 9 550 \cdot L + 1 225 \cdot (1 + 0.2 \cdot (L−1)) = 9 795 \cdot L + 980$

| Load | Baseline | Optimized | Saving |
| --- | --- | --- | --- |
| 0.5 | $12,900 | $5,878 | 54.4% |
| 1.0 | $23,000 | $10,775 | 53.2% |
| 2.0 | $43,200 | $20,570 | 52.4% |
| 3.0 | $63,400 | $30,365 | 52.1% |
| 5.0 | $103,800 | $53,855 | 51.9% |
| $\infty$ | — | —  | -> 51.5% (асимптота 9 795 / 20 200) |

Точки перелома нет. Saving монотонно убывает с ~54% до ~51.5% и остаётся кратно выше цели в 20% при любой нагрузке. Storage с ростом load «разбавляется» compute'ом, поэтому процент чуть падает — но это микроэффект.

Архитектурный вывод: правильная оптимизация - это не разовая скидка, а структурное изменение cost/load функции.